# 01 — Data Understanding & Quality Assessment
## Banking Credit Risk & Financial Analytics
**Phase 2 — Data Quality + Cleaning**

This notebook documents the complete data quality assessment for the  
UCI Default of Credit Card Clients dataset (30,000 records, 25 columns).

Every cleaning decision is documented in:
- `docs/data_quality_and_cleaning.md`
- `docs/decisions.md`

The processed output is written to `data/processed/credit_risk_clean.csv`.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_FILE = os.path.join(BASE_DIR, 'data', 'raw', 'default of credit card clients.xls')
PROCESSED_FILE = os.path.join(BASE_DIR, 'data', 'processed', 'credit_risk_clean.csv')

print('Base dir:', BASE_DIR)
print('Raw file exists:', os.path.exists(RAW_FILE))
print('Processed file exists:', os.path.exists(PROCESSED_FILE))


## 1. Load Raw Dataset

In [ ]:
# Load using row 1 as header (descriptive names)
# Row 0 contains UCI variable codes (X1-X23, Y)
df_raw = pd.read_excel(RAW_FILE, sheet_name=0, header=1)

print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print()
print('Columns:')
for i, col in enumerate(df_raw.columns):
    print(f'  [{i:2d}] {col}')


In [ ]:
# Data types
print('Data types:')
print(df_raw.dtypes)


## 2. Basic Quality Checks

In [ ]:
# Missing values
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
print(f'Total nulls: {missing.sum()}')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values in any column.')


In [ ]:
# Duplicates
print('=== Duplicates ===')
n_dup_id = df_raw['ID'].duplicated().sum()
n_dup_row = df_raw.duplicated().sum()
print(f'Duplicate IDs  : {n_dup_id}')
print(f'Duplicate rows : {n_dup_row}')


## 3. Target Variable Distribution

In [ ]:
target = 'default payment next month'
vc = df_raw[target].value_counts().sort_index()
pct = df_raw[target].value_counts(normalize=True).sort_index() * 100

print('=== Target: default payment next month ===')
print(f'  0 (No default) : {vc[0]:,} ({pct[0]:.2f}%)')
print(f'  1 (Default)    : {vc[1]:,} ({pct[1]:.2f}%)')
print()
print('Class imbalance ratio (majority/minority):', round(vc[0]/vc[1], 2))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['No Default (0)', 'Default (1)'], [vc[0], vc[1]],
              color=['#4CAF50', '#F44336'], edgecolor='white', linewidth=1.2)
for bar, count, pct_val in zip(bars, [vc[0], vc[1]], [pct[0], pct[1]]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{count:,}
({pct_val:.1f}%)', ha='center', va='bottom', fontsize=11)
ax.set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Customers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, 28000)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'data', 'processed', 'target_distribution.png'), dpi=120)
plt.show()
print('Note: 22.12% default rate - moderate class imbalance. Must handle in ML phase.')


## 4. Categorical Variables — Code Investigation

In [ ]:
# SEX
print('=== SEX ===')
print('Documented codes: 1=Male, 2=Female')
print(df_raw['SEX'].value_counts().sort_index())
print('Unexpected codes:', set(df_raw['SEX'].unique()) - {1, 2})


In [ ]:
# EDUCATION
print('=== EDUCATION ===')
print('Documented codes: 1=Graduate, 2=University, 3=High school, 4=Others')
edu_vc = df_raw['EDUCATION'].value_counts().sort_index()
print(edu_vc)
print()
undoc = df_raw['EDUCATION'].isin([0, 5, 6]).sum()
print(f'Undocumented codes (0, 5, 6): {undoc} records ({undoc/len(df_raw)*100:.2f}%)')


In [ ]:
# EDUCATION default rates by code
print('=== EDUCATION default rate by code ===')
edu_grp = df_raw.groupby('EDUCATION')[target].agg(['count', 'sum', 'mean']).round(4)
edu_grp.columns = ['count', 'defaults', 'default_rate']
print(edu_grp)
print()
print('Decision: Remap codes 0, 5, 6 -> 4 (Others). See docs/decisions.md D-003.')


In [ ]:
# MARRIAGE
print('=== MARRIAGE ===')
print('Documented codes: 1=Married, 2=Single, 3=Others')
mar_vc = df_raw['MARRIAGE'].value_counts().sort_index()
print(mar_vc)
print()
undoc = (df_raw['MARRIAGE'] == 0).sum()
print(f'Undocumented code 0: {undoc} records ({undoc/len(df_raw)*100:.2f}%)')
print()
mar_grp = df_raw.groupby('MARRIAGE')[target].agg(['count', 'sum', 'mean']).round(4)
mar_grp.columns = ['count', 'defaults', 'default_rate']
print(mar_grp)
print()
print('Decision: Remap code 0 -> 3 (Others). See docs/decisions.md D-004.')


## 5. PAY Columns — Repayment Status Code Investigation

In [ ]:
# PAY_0 distribution
print('=== PAY_0 value counts ===')
print('Documented codes: -1=Paid duly, 0=Revolving credit, 1-8=Months delayed')
print('Undocumented: -2 (present in data)')
print()
print(df_raw['PAY_0'].value_counts().sort_index())


In [ ]:
# Cross-check PAY_0 = -2 with BILL_AMT1 to understand -2 meaning
print('=== PAY_0 = -2: Cross-check with BILL_AMT1 ===')
for code in [-2, -1, 0]:
    sub = df_raw[df_raw['PAY_0'] == code]
    neg_bill = (sub['BILL_AMT1'] < 0).sum()
    zero_bill = (sub['BILL_AMT1'] == 0).sum()
    pos_bill = (sub['BILL_AMT1'] > 0).sum()
    print(f'PAY_0={code:2d} (n={len(sub):5,}): BILL_AMT1 negative={neg_bill:4}, zero={zero_bill:4}, positive={pos_bill:5}')

print()
print('Interpretation:')
print('  -2: No new consumption. Card not used OR balance cleared with no new charges.')
print('      Distinct from -1 (used and fully paid) and 0 (revolving credit active).')
print()
print('Decision: Retain -2 as valid separate category. See docs/decisions.md D-005.')


In [ ]:
# Summary of PAY -2 counts across all PAY columns
print('=== PAY code -2 counts across all columns ===')
for col in ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']:
    n = (df_raw[col] == -2).sum()
    pct = n / len(df_raw) * 100
    print(f'  {col}: {n:,} records ({pct:.1f}%)')

print()
print('Note: PAY_5 and PAY_6 have no value of 1 (1-month delay) in this dataset.')


## 6. Financial Columns — Numerical Investigation

In [ ]:
# AGE
print('=== AGE ===')
print(df_raw['AGE'].describe(percentiles=[.1, .25, .5, .75, .9, .99]))
print()
print('Age > 70:', (df_raw['AGE'] > 70).sum())
print('Age == 21 (minimum):', (df_raw['AGE'] == 21).sum())
print('Implausible ages (< 18 or > 100):', ((df_raw['AGE'] < 18) | (df_raw['AGE'] > 100)).sum())
print()
print('Decision: Retain all ages. Range 21-79 is plausible for Taiwanese credit card holders.')


In [ ]:
# LIMIT_BAL
print('=== LIMIT_BAL (Credit Limit, NT$) ===')
print(df_raw['LIMIT_BAL'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
print()
print('Unique credit limit values:', df_raw['LIMIT_BAL'].nunique())
print('Zero or negative:', (df_raw['LIMIT_BAL'] <= 0).sum())
print()
print('Decision: Retain all. No implausible values found.')


In [ ]:
# BILL_AMT columns - negatives investigation
print('=== BILL_AMT columns — negative values ===')
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
months = ['Sep-05', 'Aug-05', 'Jul-05', 'Jun-05', 'May-05', 'Apr-05']

for col, month in zip(bill_cols, months):
    neg = (df_raw[col] < 0).sum()
    zero = (df_raw[col] == 0).sum()
    mn = df_raw[col].min()
    mx = df_raw[col].max()
    pct_neg = neg / len(df_raw) * 100
    print(f'  {col} ({month}): min={mn:,}  max={mx:,}  negatives={neg} ({pct_neg:.1f}%)  zeros={zero}')

print()
print('Decision: Negative bill amounts = credit balances (overpayment/refunds). Valid, retain.')
print('See docs/decisions.md D-006.')


In [ ]:
# PAY_AMT columns
print('=== PAY_AMT columns — payment amounts ===')
pay_amt_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

for col in pay_amt_cols:
    neg = (df_raw[col] < 0).sum()
    zero = (df_raw[col] == 0).sum()
    pct_zero = zero / len(df_raw) * 100
    mx = df_raw[col].max()
    print(f'  {col}: min={df_raw[col].min()}  max={mx:,}  negatives={neg}  zeros={zero} ({pct_zero:.1f}%)')

print()
print('No negative payment amounts found (as expected).')
print('Zero payment amounts are valid: customers who made no payment that month.')


## 7. Apply Cleaning and Export Processed Dataset

In [ ]:
# Apply all cleaning decisions
df = df_raw.copy()

# Rename target
df = df.rename(columns={'default payment next month': 'DEFAULT'})

# Remap EDUCATION undocumented codes -> 4 (Others)
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# Remap MARRIAGE undocumented code -> 3 (Others)
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

# No other modifications needed
print('Cleaning applied.')
print()
print('=== Validation ===')
print(f'Shape         : {df.shape}')
print(f'Nulls         : {df.isnull().sum().sum()}')
print(f'EDUCATION     : {sorted(df["EDUCATION"].unique())}')
print(f'MARRIAGE      : {sorted(df["MARRIAGE"].unique())}')
print(f'SEX           : {sorted(df["SEX"].unique())}')
print(f'DEFAULT       : {sorted(df["DEFAULT"].unique())}')
print(f'DEFAULT rate  : {df["DEFAULT"].mean():.4f}')


In [ ]:
# Export
df.to_csv(PROCESSED_FILE, index=False)
file_size = os.path.getsize(PROCESSED_FILE) / (1024 * 1024)
print(f'Processed dataset exported: {PROCESSED_FILE}')
print(f'File size: {file_size:.2f} MB')
print()
print('=== Cleaning Summary ===')
print(f'  Original rows         : {df_raw.shape[0]:,}')
print(f'  Processed rows        : {df.shape[0]:,}  (no rows dropped)')
print(f'  EDUCATION remapped    : 345 records (codes 0,5,6 -> 4)')
print(f'  MARRIAGE remapped     : 54 records (code 0 -> 3)')
print(f'  PAY -2 retained       : {(df["PAY_0"]==-2).sum():,} records in PAY_0 alone')
print(f'  BILL_AMT negatives    : {(df["BILL_AMT1"]<0).sum()} in BILL_AMT1 (retained, valid)')
print(f'  Rows dropped          : 0')
print(f'  New missing values    : 0')


## 8. Cleaned Data Summary

In [ ]:
# Final summary of the processed dataset
print('=== Cleaned Dataset — Column Summary ===')
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'nulls': df.isnull().sum(),
    'unique': df.nunique(),
    'min': df.min(),
    'max': df.max(),
    'mean': df.mean().round(2)
})
print(summary.to_string())


## 9. Data Quality Decisions Summary

| Decision | Issue | Treatment | Records Affected |
|---|---|---|---|
| D-001 | File format (double-header XLS) | Load with `header=1` | All 30,000 |
| D-002 | Target column name with spaces | Rename to `DEFAULT` | All 30,000 |
| D-003 | EDUCATION codes 0, 5, 6 undocumented | Remap to 4 (Others) | 345 (1.15%) |
| D-004 | MARRIAGE code 0 undocumented | Remap to 3 (Others) | 54 (0.18%) |
| D-005 | PAY code -2 undocumented | Retain as valid category | ~2,759–4,895 per column |
| D-006 | Negative BILL_AMT values | Retain as credit balances | ~590–688 per column |
| D-007 | PAY_AMT zero values | Retain (no payment made) | ~5,249–7,173 per column |
| D-008 | AGE range (21–79) | Retain all (plausible) | 0 changes |
| D-009 | LIMIT_BAL range (10K–1M NT$) | Retain all (plausible) | 0 changes |

**Rows dropped: 0** — No records were removed from the dataset.
All data quality issues were resolved through remapping or documented retention decisions.
